# 02 · Checking the preprocessed views

Run `python -m src.preprocess` first. This notebook checks that the saved views are what the CNN should see,
**before** any training. Bad views can't be fixed by a better model.

In [ ]:
import sys, os
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "download.py").exists())
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

P = ROOT / "data" / "processed_full"
G, L = np.load(P / "global.npy"), np.load(P / "local.npy")
labels = pd.read_csv(P / "labels.csv")
failed = pd.read_csv(P / "failed.csv")
COLORS = {"PC": "tab:green", "AFP": "tab:red", "NTP": "tab:gray"}
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3})

print("global:", G.shape, " local:", L.shape, " labels:", len(labels))
print("NaNs:", int(np.isnan(G).sum() + np.isnan(L).sum()))
print(labels.av_training_set.value_counts().to_string())

**What you should see:** the same number of rows in all three (15,730 for the full dataset) and **0 NaNs**.

## 1 · What failed, and why

In [ ]:
print(f"{len(failed)} TCEs failed")
failed.groupby(["av_training_set", "error"]).size().sort_values(ascending=False).head(15)

**What you should see:** only a handful of failures, ideally under 1%. If one class fails much more than the others,
the model will see a biased sample, so look at those TCEs in notebook 01 with `show_tce`.

## 2 · The average view per class

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4))
xg = np.linspace(-0.5, 0.5, G.shape[1]); xl = np.linspace(-2, 2, L.shape[1])
for cls in ["NTP", "AFP", "PC"]:
    m = (labels.av_training_set == cls).to_numpy()
    ax[0].plot(xg, np.median(G[m], axis=0), color=COLORS[cls], label=cls, lw=1)
    ax[1].plot(xl, np.median(L[m], axis=0), color=COLORS[cls], label=cls, lw=2)
ax[0].set(title="median global view", xlabel="phase [orbits]"); ax[0].legend()
ax[1].set(title="median local view", xlabel="phase [transit durations]")
plt.tight_layout()

**What you should see:**

- **PC and AFP:** a clear dip at 0 reaching about −1, flat elsewhere. The AFP local dip is usually more **V-shaped**
  (pointed bottom), the PC dip more **U-shaped** (flatter bottom).
- **AFP global view:** often a small extra dip at ±0.5 orbits, from secondary eclipses.
- **NTP:** close to flat, perhaps a small dip at 0. These signals are mostly noise, so the average washes out.

If the dips aren't centred at 0, the folding is off. If they're much shallower than −1, the normalisation is off.

## 3 · Random examples

In [ ]:
def grid(cls, n=6, seed=0):
    idx = labels.index[labels.av_training_set == cls].to_series().sample(n, random_state=seed).to_numpy()
    fig, ax = plt.subplots(2, n, figsize=(2.8 * n, 4.6))
    for j, i in enumerate(idx):
        r = labels.loc[i]
        ax[0, j].plot(xg, G[i], color=COLORS[cls], lw=0.5)
        ax[0, j].set_title(f"{r.kepid} · {r.tce_period:.1f} d", fontsize=8)
        ax[1, j].plot(xl, L[i], color=COLORS[cls], lw=1.2)
        for a in ax[:, j]: a.set_yticks([-1, 0]); a.tick_params(labelsize=7)
    ax[0, 0].set_ylabel("global"); ax[1, 0].set_ylabel("local")
    fig.suptitle(cls); plt.tight_layout()

for cls in ["PC", "AFP", "NTP"]:
    grid(cls)

**What you should see:** most PCs and AFPs with a clean dip in the middle of both views. Some weak signals
look like pure noise. That's expected: normalising stretches whatever is deepest to −1, even if it's noise.
Change `seed` to see others. A few strange ones are fine; many identical-looking failures are a bug.

## 4 · The split

In [ ]:
split = pd.read_csv(P / "split.csv")
m = labels.merge(split, on=["kepid", "tce_plnt_num"], how="left")
print("views without a split:", m.split.isna().sum(), "(must be 0)")
m.groupby("split").agg(n=("label", "size"), planet_share=("label", "mean")).round(3)

**What you should see:** roughly 80/10/10 with a similar planet share (about 23% for the full dataset) in each split, and 0 views
without a split.